In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/IKEA_product_catalog_MX.csv")
df.head()

,unique_id,product_id,product_name,product_type,product_measurements,product_description,main_category,sub_category,product_rating,product_rating_count,badge,online_sellable,url,price,currency,discount,sale_tag
0,10018194-Mexico,10018194,ORDNING,Dish drainer,50x27x36 cm,"ORDNING Dish drainer, stainless steel, 50x27x3...",cookware-tableware,dishwashing-accessories,4.4,24.0,none,True,https://www.ikea.com/mx/en/p/ordning-dish-drai...,699.0,MXN,none,NONE
1,10056770-Mexico,10056770,DIMPA,Storage bag,65x22x65 cm,"DIMPA Storage bag, transparent, 65x22x65 cm",small-storage-organisers,clothes-organisers,4.8,75.0,none,True,https://www.ikea.com/mx/en/p/dimpa-storage-bag...,99.0,MXN,none,NONE
2,10069763-Mexico,10069763,OMAR,Shelving unit,92x36x94 cm,"OMAR Shelving unit, galvanised, 92x36x94 cm",storage-organisation,bookcases-shelving-units,4.8,71.0,none,True,https://www.ikea.com/mx/en/p/omar-shelving-uni...,1199.0,MXN,none,NONE
3,10091453-Mexico,10091453,TROFAST,Frame,99x44x94 cm,"TROFAST Frame, white, 99x44x94 cm",storage-organisation,storage-solution-systems,4.8,30.0,none,True,https://www.ikea.com/mx/en/p/trofast-frame-whi...,1699.0,MXN,none,NONE
4,10092216-Mexico,10092216,MJÖD,Beer tankard,60 cl,"MJÖD Beer tankard, clear glass, 60 cl",cookware-tableware,glassware-jugs,5.0,3.0,none,True,https://www.ikea.com/mx/en/p/mjoed-beer-tankar...,69.0,MXN,none,NONE


In [2]:
df.shape

(7475, 17)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7475 entries, 0 to 7474
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   unique_id             7475 non-null   str    
 1   product_id            7475 non-null   int64  
 2   product_name          7475 non-null   str    
 3   product_type          7475 non-null   str    
 4   product_measurements  7475 non-null   str    
 5   product_description   7475 non-null   str    
 6   main_category         7475 non-null   str    
 7   sub_category          7475 non-null   str    
 8   product_rating        7475 non-null   str    
 9   product_rating_count  7475 non-null   str    
 10  badge                 7475 non-null   str    
 11  online_sellable       7475 non-null   bool   
 12  url                   7475 non-null   str    
 13  price                 7475 non-null   float64
 14  currency              7475 non-null   str    
 15  discount              7475 non-n

## Type Casting and Feature Selection

### Design Decisions and ETL Criteria

The following transformations will be applied to the respective columns based on previous analysis:
- `"main_category"` from "str" to "category" data type.
- `"badge"` from "str" to "category" data type.
- `"sale_tag"` from "str" to "category" data type.
- `"product_id"` from "int64" to "str" data type.
- `"product_rating"` from "str" to "float32" data type.
- `"product_rating_count"` from "str" to "float32" data type.
- `"discount"` from "str" to "float32" data type.
- `"currency"` column dropped.

To enable the previous transformations the following changes will be applied:
- "none" values transformed to NaN values for the following columns: `"product_rating", "product_rating_count", "discount"`
  
Transforming "none" string values to NaN values can reduce the dataset's size, enabling better calculations in numerical columns for analysis in BI tools.

#### Feature Selection
Being a subset for a single country and having a single currency, the `"currency"` column does not contribute any relevancy for the dataset. Removing it might reduce memory consumption and makes the dataset cleaner to process in Power BI.

The `"discount"` column for this subset contains only 'none' values, but removing such column might imply future problems calculating discounts. To ensure the correct implementation and analysis with BI tools, this column will be normalized to null values (NaN) and transformed to float32 data type. Analyzing the full worldwide dataset we observed a standard format of data (i.e. "Discount 10%), this data can be automatically transformed to percentages as decimal points, enabling a better implementation for BI tools and later numerical analysis.

After implementing all transformations there was a memory usage reduction from 2.6 mb to 2.1mb (~13%). Memory reduction is crucial in big datasets, allowing for better performance in operations, scalability and less RAM consumption.

In [4]:
# replace "none" values for product ratings, counts and currency for data transformation.

df["product_rating"] = df["product_rating"].replace("none",np.nan)
df["product_rating_count"] = df["product_rating_count"].replace("none",np.nan)
df["discount"] = df["discount"].replace("none",np.nan)

# cast data type transformations
df = df.astype({
    "product_id": "str",
    "main_category": "category",
    "badge": "category",
    "sale_tag": "category",
    "product_rating": "float32",
    "product_rating_count": "float32",
    "discount": "float32"
})

# drop columns
df = df.drop(columns=["currency"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7475 entries, 0 to 7474
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   unique_id             7475 non-null   str     
 1   product_id            7475 non-null   str     
 2   product_name          7475 non-null   str     
 3   product_type          7475 non-null   str     
 4   product_measurements  7475 non-null   str     
 5   product_description   7475 non-null   str     
 6   main_category         7475 non-null   category
 7   sub_category          7475 non-null   str     
 8   product_rating        5447 non-null   float32 
 9   product_rating_count  5447 non-null   float32 
 10  badge                 7475 non-null   category
 11  online_sellable       7475 non-null   bool    
 12  url                   7475 non-null   str     
 13  price                 7475 non-null   float64 
 14  discount              0 non-null      float32 
 15  sale_tag       

In [5]:
df.head(2)

,unique_id,product_id,product_name,product_type,product_measurements,product_description,main_category,sub_category,product_rating,product_rating_count,badge,online_sellable,url,price,discount,sale_tag
0,10018194-Mexico,10018194,ORDNING,Dish drainer,50x27x36 cm,"ORDNING Dish drainer, stainless steel, 50x27x3...",cookware-tableware,dishwashing-accessories,4.4,24.0,none,True,https://www.ikea.com/mx/en/p/ordning-dish-drai...,699.0,NaN,NONE
1,10056770-Mexico,10056770,DIMPA,Storage bag,65x22x65 cm,"DIMPA Storage bag, transparent, 65x22x65 cm",small-storage-organisers,clothes-organisers,4.8,75.0,none,True,https://www.ikea.com/mx/en/p/dimpa-storage-bag...,99.0,NaN,NONE


## Save Dataset

In [6]:
df.to_csv("../data/IKEA_mexico_clean.csv", index=False)